# NB03 — Embedding Retrieval

Fetch ESM-2 embeddings for all Swiss-Prot candidate proteins via the `llm_homology_api`.
Verify that pre-computed genome FAISS indexes load correctly.

**Inputs**: `data/all_candidates.parquet`, `user_data/ecoli-files/` (genome FAISS indexes)

**Outputs**: `data/candidate_embeddings.json`, `data/candidate_faiss.index`, `data/candidate_ids.json`

In [1]:
import os
import time
import json
import asyncio
import httpx
import numpy as np
import faiss
import pandas as pd
from pathlib import Path

BASE_URL = 'https://kbase.us/services/llm_homology_api'
SEARCH_ENDPOINT = f'{BASE_URL}/search'
RESULT_ENDPOINT = f'{BASE_URL}/result'
BATCH_SIZE = 500
POLL_INTERVAL = 5.0

USER_DATA = Path('..') / 'user_data'
DATA_OUT = Path('..') / 'user_data'
GENOME_DIR = USER_DATA / 'ecoli-files'
FASTA_EXT = '.faa'

## 1. Load Candidate Proteins

In [2]:
candidates = pd.read_parquet(DATA_OUT / 'all_candidates.parquet')
unique_proteins = candidates.drop_duplicates(subset='uniprot_id')[['uniprot_id', 'sequence']]

print(f"Total candidate pairs: {len(candidates)}")
print(f"Unique proteins to embed: {len(unique_proteins)}")
print(f"Batches needed: {(len(unique_proteins) // BATCH_SIZE) + 1}")
print(f"Sequence length range: {unique_proteins['sequence'].str.len().min()}-{unique_proteins['sequence'].str.len().max()} aa")

Total candidate pairs: 4123
Unique proteins to embed: 3168
Batches needed: 7
Sequence length range: 15-2753 aa


## 2. Submit Embedding Jobs

In [3]:
query_list = [
    {'id': row['uniprot_id'], 'sequence': row['sequence']}
    for _, row in unique_proteins.iterrows()
]

session_jobs = []
last_api_time = 0.0

async def submit_jobs(queries):
    jobs = []
    async with httpx.AsyncClient(timeout=120.0) as client:
        for i in range(0, len(queries), BATCH_SIZE):
            batch = queries[i:i+BATCH_SIZE]
            payload = {
                "query_sequences": batch,
                "similarity_threshold": 1.0,
                "best_hit_only": True,
                "max_hits": 1,
                "return_query_embeddings": True,
                "return_hit_embeddings": False
            }
            resp = await client.post(SEARCH_ENDPOINT, json=payload)
            resp.raise_for_status()
            job_id = resp.json().get('job_id')
            if job_id:
                jobs.append(job_id)
                print(f"  Batch {i//BATCH_SIZE + 1}/{(len(queries)//BATCH_SIZE)+1}: job {job_id}")
    return jobs

print(f"Submitting {len(query_list)} proteins in batches of {BATCH_SIZE}...")
session_jobs = await submit_jobs(query_list)
last_api_time = time.time()
print(f"\nSubmitted {len(session_jobs)} jobs.")

Submitting 3168 proteins in batches of 500...
  Batch 1/7: job 68de5da1-5f50-4c54-a9ca-79dd30f38677


  Batch 2/7: job 85ed7d6e-7567-4c27-91f8-bb923fe74339


  Batch 3/7: job e97ccdbe-c880-491d-9682-77d96c209016


  Batch 4/7: job 197dd952-98ba-4b36-a10e-dc5589de00e8


  Batch 5/7: job 39e73609-bdf0-45f7-b915-bda9b6d8bcee


  Batch 6/7: job 010a65d1-28e5-49b6-8ac0-828c141175e5


  Batch 7/7: job fcd2e2b1-9af9-4be5-94fc-0f553db3e889

Submitted 7 jobs.


## 3. Retrieve Embeddings

In [4]:
async def fetch_results(jobs):
    global last_api_time
    all_results = []

    async with httpx.AsyncClient(timeout=120.0) as client:
        for job_id in jobs:
            while True:
                elapsed = time.time() - last_api_time
                if elapsed < 5.0:
                    await asyncio.sleep(5.0 - elapsed)

                resp = await client.post(RESULT_ENDPOINT, json={"job_id": job_id})
                resp.raise_for_status()
                data = resp.json()
                status = data.get('status')
                last_api_time = time.time()

                if status == 'done':
                    print(f"  Job {job_id}: DONE")
                    res_data = data.get('result', [])
                    if isinstance(res_data, dict) and 'hits' in res_data:
                        all_results.extend(res_data['hits'])
                    elif isinstance(res_data, list):
                        all_results.extend(res_data)
                    else:
                        all_results.append(res_data)
                    break
                elif status in ('failed', 'error'):
                    print(f"  Job {job_id}: FAILED — {data.get('error')}")
                    break
                elif status == 'pending':
                    print(f"  Job {job_id}: pending...")
                    await asyncio.sleep(5.0)
                    last_api_time = time.time()

    return all_results

print(f"Fetching results for {len(session_jobs)} jobs...")
raw_results = await fetch_results(session_jobs)

embedding_data = []
for res in raw_results:
    if isinstance(res, dict):
        qid = res.get('query_id')
        qemb = res.get('query_embedding')
        if qid and qemb:
            embedding_data.append({'query_id': qid, 'query_embedding': qemb})

elapsed_total = time.time() - last_api_time
print(f"\nEmbeddings retrieved: {len(embedding_data)} / {len(unique_proteins)}")

missing_emb = set(unique_proteins['uniprot_id']) - {e['query_id'] for e in embedding_data}
if missing_emb:
    print(f"Missing embeddings: {len(missing_emb)}")

Fetching results for 7 jobs...


  Job 68de5da1-5f50-4c54-a9ca-79dd30f38677: DONE


  Job 85ed7d6e-7567-4c27-91f8-bb923fe74339: DONE


  Job e97ccdbe-c880-491d-9682-77d96c209016: DONE


  Job 197dd952-98ba-4b36-a10e-dc5589de00e8: DONE


  Job 39e73609-bdf0-45f7-b915-bda9b6d8bcee: DONE


  Job 010a65d1-28e5-49b6-8ac0-828c141175e5: DONE


  Job fcd2e2b1-9af9-4be5-94fc-0f553db3e889: DONE

Embeddings retrieved: 3168 / 3168


## 4. Save Embeddings and Build FAISS Index

In [5]:
with open(DATA_OUT / 'candidate_embeddings.json', 'w') as f:
    json.dump(embedding_data, f)
print(f"Saved {len(embedding_data)} embeddings to data/candidate_embeddings.json")

if not embedding_data:
    print("No embeddings retrieved — skipping FAISS index build.")
    print("Re-run this notebook when the llm_homology_api is available.")
    dim = None
else:
    ids = [e['query_id'] for e in embedding_data]
    vectors = np.array([e['query_embedding'] for e in embedding_data], dtype='float32')
    faiss.normalize_L2(vectors)

    dim = vectors.shape[1]
    index = faiss.IndexFlatIP(dim)
    index.add(vectors)

    faiss.write_index(index, str(DATA_OUT / 'candidate_faiss.index'))
    with open(DATA_OUT / 'candidate_ids.json', 'w') as f:
        json.dump(ids, f)

    print(f"FAISS index: {index.ntotal} vectors, dim={dim}")
    print(f"Saved to data/candidate_faiss.index + data/candidate_ids.json")

Saved 3168 embeddings to data/candidate_embeddings.json
FAISS index: 3168 vectors, dim=1280
Saved to data/candidate_faiss.index + data/candidate_ids.json


## 5. Verify Genome FAISS Indexes

In [6]:
genome_files = sorted([
    f.replace(f'-protein{FASTA_EXT}.faiss', '')
    for f in os.listdir(GENOME_DIR)
    if f.endswith(f'{FASTA_EXT}.faiss') and not f.startswith('._')
])
print(f"Genome FAISS indexes found: {len(genome_files)}")

test_genome = genome_files[0]
test_index = faiss.read_index(str(GENOME_DIR / f'{test_genome}-protein{FASTA_EXT}.faiss'))
with open(GENOME_DIR / f'{test_genome}-protein{FASTA_EXT}.results.json') as f:
    test_ids = json.load(f)

genome_dim = test_index.d
print(f"\nTest genome: {test_genome}")
print(f"  Index vectors: {test_index.ntotal}")
print(f"  Index dimension: {genome_dim}")
print(f"  ID mapping entries: {len(test_ids)}")

if dim and genome_dim != dim:
    print(f"  WARNING: dimension mismatch! Candidate dim={dim}, genome dim={genome_dim}")
elif dim:
    print(f"  Dimensions match ({dim}d). Cross-search is compatible.")
else:
    print(f"  Candidate embeddings not yet available — dimension check deferred.")

Genome FAISS indexes found: 48



Test genome: 219790_10_1
  Index vectors: 4518
  Index dimension: 1280
  ID mapping entries: 4518
  Dimensions match (1280d). Cross-search is compatible.


In [7]:
print("Validating all genome indexes...")
errors = []
genome_stats = []
for g in genome_files:
    faiss_path = GENOME_DIR / f'{g}-protein{FASTA_EXT}.faiss'
    json_path = GENOME_DIR / f'{g}-protein{FASTA_EXT}.results.json'

    if not faiss_path.exists():
        errors.append(f"{g}: missing FAISS index")
        continue
    if not json_path.exists():
        errors.append(f"{g}: missing ID mapping")
        continue

    idx = faiss.read_index(str(faiss_path))
    with open(json_path) as f:
        gids = json.load(f)

    if idx.d != genome_dim:
        errors.append(f"{g}: dim mismatch ({idx.d} vs {genome_dim})")
    if idx.ntotal != len(gids):
        errors.append(f"{g}: index/ID count mismatch ({idx.ntotal} vs {len(gids)})")

    genome_stats.append({'genome': g, 'n_proteins': idx.ntotal, 'dim': idx.d})

stats_df = pd.DataFrame(genome_stats)
print(f"\nValidated {len(genome_stats)} genomes")
print(f"Proteins per genome: {stats_df['n_proteins'].min()}-{stats_df['n_proteins'].max()} (mean {stats_df['n_proteins'].mean():.0f})")

if errors:
    print(f"\nERRORS ({len(errors)}):")
    for e in errors:
        print(f"  {e}")
else:
    print("All genome indexes valid and compatible.")

Validating all genome indexes...



Validated 48 genomes
Proteins per genome: 3970-5207 (mean 4604)
All genome indexes valid and compatible.


## 6. Quick Sanity Check

Search one candidate against one genome to verify the cross-search works.

In [8]:
if embedding_data:
    test_vec = np.array([embedding_data[0]['query_embedding']], dtype='float32')
    faiss.normalize_L2(test_vec)

    D, I = test_index.search(test_vec, 5)

    print(f"Query: {embedding_data[0]['query_id']}")
    print(f"Genome: {test_genome}")
    print(f"Top-5 hits:")
    for rank, (dist, idx) in enumerate(zip(D[0], I[0])):
        if idx >= 0 and idx < len(test_ids):
            gene_entry = test_ids[idx]
            gene_id = gene_entry.get('query_id', gene_entry) if isinstance(gene_entry, dict) else gene_entry
            print(f"  {rank+1}. {gene_id} (cosine sim: {dist:.4f})")

    print(f"\nReady for NB04: similarity scoring across {len(genome_files)} genomes.")
else:
    print("No embeddings available — cannot run sanity check.")
    print("The llm_homology_api may be down. Re-run this notebook when it is available.")

Query: O62768
Genome: 219790_10_1
Top-5 hits:
  1. 1986 (cosine sim: 0.9768)
  2. 1739 (cosine sim: 0.9747)
  3. 3199 (cosine sim: 0.9733)
  4. 4135 (cosine sim: 0.9732)
  5. 1266 (cosine sim: 0.9730)

Ready for NB04: similarity scoring across 48 genomes.
